# TinyGPT v2 — Dual-Mix SFT Notebook

Runs Mix A (70% chat / 30% Q&A) and Mix B (30% chat / 70% Q&A) from the same untouched Stage 3 checkpoint, then compares Base vs both SFT models.

**Important:** Run All. The notebook validates dataset availability, checkpoint compatibility, token budgets, and response masking before training. It refuses to silently upsample data. The exact usable data counts are discovered at runtime.

## Cell 1 — Setup, checkpoint, notifications, and baseline

In [1]:
import os, gc, json, time, math, random, hashlib, traceback
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Config, GPT2LMHeadModel, AutoTokenizer, get_cosine_schedule_with_warmup

CFG = {
    "base": "/kaggle/input/datasets/fakesiddharth/base-pretrained-model/checkpoints/real_best.pt",
    "out": "/kaggle/working/tinygpt_sft",
    "ntfy": "tinygpt-siddharth-48271",
    "seed": 20261017, "context": 256,
    "target_active_tokens_epoch": 1_200_000,
    "min_active_tokens_epoch": 100_000,
    "epochs": 2, "per_gpu_batch": 16, "grad_accum": 2,
    "lr": 5e-5, "min_lr": 5e-6, "warmup_fraction": 0.05,
    "weight_decay": 0.01, "grad_clip": 1.0,
    "dropout": 0.0, "log_every": 25, "validate_every": 100,
    "max_new_tokens": 80, "seeds": [11, 29, 47, 71],
}
os.makedirs(CFG["out"], exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda": raise RuntimeError("Enable a Kaggle GPU accelerator before running.")
print("PyTorch:", torch.__version__, "| device:", device, "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), f"{torch.cuda.get_device_properties(i).total_memory/1024**3:.1f} GiB")

def notify(title, msg):
    try:
        import requests
        requests.post("https://ntfy.sh/"+CFG["ntfy"], data=str(msg).encode(),
                      headers={"Title":str(title).encode("ascii","replace").decode()[:100]},
                      timeout=5)
    except Exception as e: print("[ntfy skipped]", type(e).__name__, str(e)[:120])

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_all(CFG["seed"])
def locate_file(preferred, basename):
    if os.path.isfile(preferred): return preferred
    found=[]
    for root,_,files in os.walk("/kaggle/input"):
        if basename in files: found.append(os.path.join(root,basename))
    if not found: raise FileNotFoundError(f"Attach {basename} as a Kaggle input.")
    found.sort(key=lambda p: ("/base-pretrained-model/" not in p, len(p)))
    return found[0]

BASE_PATH=locate_file(CFG["base"],"real_best.pt")
print("Base:", BASE_PATH, f"{os.path.getsize(BASE_PATH)/1024**2:.1f} MiB")
tokenizer=AutoTokenizer.from_pretrained("gpt2")
EOS=tokenizer.eos_token_id
assert len(tokenizer)==50257 and EOS==50256

MC=GPT2Config(vocab_size=50257,n_positions=256,n_ctx=256,n_embd=512,n_layer=8,n_head=8,
              resid_pdrop=CFG["dropout"],embd_pdrop=CFG["dropout"],attn_pdrop=CFG["dropout"],
              bos_token_id=EOS,eos_token_id=EOS,pad_token_id=EOS,use_cache=False)

def get_state(path=BASE_PATH):
    obj=torch.load(path,map_location="cpu",weights_only=False)
    if isinstance(obj,dict):
        for k in ("model_state_dict","state_dict","model","weights"):
            if k in obj and isinstance(obj[k],dict): obj=obj[k]; break
    if not isinstance(obj,dict): raise TypeError(f"Unexpected checkpoint type: {type(obj)}")
    if obj and all(k.startswith("module.") for k in obj): obj={k[7:]:v for k,v in obj.items()}
    return obj

def new_model(state=None):
    m=GPT2LMHeadModel(MC)
    missing,unexpected=m.load_state_dict(state if state is not None else get_state(),strict=False)
    if missing or unexpected: raise RuntimeError(f"Checkpoint mismatch. Missing={missing[:5]}, unexpected={unexpected[:5]}")
    if sum(p.numel() for p in m.parameters()) != 51_082_752: raise RuntimeError("Unexpected parameter count.")
    return m

def cache_path():
    for root,_,files in os.walk("/kaggle/input"):
        if "validation_cache.pt" in files: return os.path.join(root,"validation_cache.pt")
    return None

def normalize_cache(obj):
    if torch.is_tensor(obj): candidates=[obj]
    elif isinstance(obj,dict): candidates=[obj[k] for k in ("input_ids","validation_ids","tokens","cache","sequences","val_ids") if k in obj]
    else: candidates=[]
    for x in candidates:
        if not torch.is_tensor(x): continue
        x=x.detach().cpu().long()
        if x.ndim==1 and x.numel()>=256: return x[:(x.numel()//256)*256].view(-1,256)
        if x.ndim==2 and x.shape[1]==256: return x
    return None

def rebuild_cache():
    from datasets import load_dataset
    print("No usable validation_cache.pt; rebuilding a deterministic FineWeb-Edu cache (may take time).")
    ds=load_dataset("HuggingFaceFW/fineweb-edu",name="sample-10BT",split="train",streaming=True,
                    revision="87f09149ef4734204d70ed1d046ddc9ca3f2b8f9").shuffle(seed=20261020,buffer_size=10000)
    chunks=[]; carry=[]; positions=0
    for row in ds:
        txt=row.get("text","")
        if not txt: continue
        h=hashlib.md5(txt.encode("utf-8",errors="replace")).hexdigest()
        if int(h[:8],16)%100 >= 5: continue
        carry.extend(tokenizer.encode(txt,add_special_tokens=False)+[EOS])
        while len(carry)>=256 and positions<250000:
            chunks.append(carry[:256]); carry=carry[256:]; positions+=256
        if positions>=250000: break
    if not chunks: raise RuntimeError("Could not rebuild FineWeb cache.")
    return torch.tensor(chunks,dtype=torch.long)

cp=cache_path()
FINEWEB=None
if cp:
    try: FINEWEB=normalize_cache(torch.load(cp,map_location="cpu",weights_only=False))
    except Exception as e: print("Cache read failed:",repr(e))
if FINEWEB is None: FINEWEB=rebuild_cache()
print("FineWeb cache:",tuple(FINEWEB.shape))

def fineweb_loss(model, batch_size=16):
    model.eval(); total=0.; count=0
    with torch.inference_mode():
        for ids in FINEWEB.split(batch_size):
            ids=ids.to(device)
            with torch.autocast("cuda",dtype=torch.float16):
                out=model(input_ids=ids,labels=ids,use_cache=False,return_dict=True)
            n=ids.shape[0]*(ids.shape[1]-1)
            total+=float(out.loss)*n; count+=n
    return total/max(count,1)

base_model=new_model().to(device)
base_loss=fineweb_loss(base_model)
print(f"Baseline FineWeb loss={base_loss:.4f}; historical target=3.7688")
if abs(base_loss-3.7688)>0.15:
    print("WARNING: baseline differs from 3.7688; check validation-cache identity before interpreting drift.")
BASE_STATE={k:v.detach().cpu().clone() for k,v in base_model.state_dict().items()}
notify("TinyGPT SFT started",f"Checkpoint loaded; FineWeb loss={base_loss:.4f}")


PyTorch: 2.11.0+cu128 | device: cuda | GPUs: 2
0 Tesla T4 14.6 GiB
1 Tesla T4 14.6 GiB
Base: /kaggle/input/notebooks/fakesiddharth/pretrain/checkpoints/real_best.pt 293.1 MiB


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2827 > 1024). Running this sequence through the model will result in indexing errors


FineWeb cache: (977, 256)


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Baseline FineWeb loss=3.6440; historical target=3.7688


## Cell 2 — Dataset loading, cleaning, tokenization, shared validation, and A/B mix construction

In [2]:
from datasets import load_dataset
import re, unicodedata

# Store full dialogue units first. Each unit remains indivisible during the train/validation split.
chat_dialogues = []
qa_dialogues = []

def try_load_parquet(repo, pattern, split="train"):
    """Load parquet directly so legacy dataset scripts are never invoked."""
    try:
        ds = load_dataset("parquet", data_files=pattern, split=split)
        print(f"Loaded parquet: {repo} | columns={ds.column_names} | rows={len(ds)}")
        return ds
    except Exception as e:
        print(f"Parquet load failed for {repo}: {type(e).__name__}: {str(e)[:220]}")
        return None

def try_load(repo, split=None, **kwargs):
    try:
        ds = load_dataset(repo, split=split, **kwargs) if split else load_dataset(repo, **kwargs)
        print("Loaded", repo, "columns=", ds.column_names, "rows=", len(ds) if hasattr(ds,"__len__") else "stream")
        return ds
    except Exception as e:
        print("FAILED", repo, type(e).__name__, str(e)[:220])
        return None

def add_dialogue(target, turns, source, conversation_id):
    """turns is a list of (user, assistant) pairs from one original conversation."""
    clean_turns=[]
    for u,a in turns:
        u=str(u).strip(); a=str(a).strip()
        if u and a:
            clean_turns.append((u,a))
    if clean_turns:
        target.append({"turns":clean_turns, "source":source, "conversation_id":str(conversation_id)})

# 1) DailyDialog: Parquet-first loading for modern datasets versions.
dd_pattern="hf://datasets/daily_dialog@refs/convert/parquet/default/train/*.parquet"
dd=try_load_parquet("daily_dialog",dd_pattern)
if dd is None:
    dd=try_load_parquet("li2017dailydialog/daily_dialog",
        "hf://datasets/li2017dailydialog/daily_dialog@refs/convert/parquet/default/train/*.parquet")
if dd is None:
    # Last fallback: some mirror revisions still expose a normal train split.
    dd=try_load("li2017dailydialog/daily_dialog",split="train")
if dd is None:
    dd=try_load_parquet("roskoN/dailydialog",
        "hf://datasets/roskoN/dailydialog@refs/convert/parquet/default/train/*.parquet")
if dd is not None:
    for idx,r in enumerate(dd):
        turns=r.get("dialog",r.get("dialogue",r.get("utterances",[])))
        if isinstance(turns,str): continue
        turns=[str(x).strip() for x in turns if str(x).strip()]
        pairs=[(turns[k],turns[k+1]) for k in range(0,len(turns)-1,2)]
        add_dialogue(chat_dialogues,pairs,"daily_dialog",f"daily_dialog:{idx}")

# 2) Everyday conversations: preserve message history within each source record.
ec=try_load("HuggingFaceTB/everyday-conversations-llama3.1-2k",split="train_sft")
if ec is None:
    # Some dataset revisions expose the default train split instead.
    ec=try_load("HuggingFaceTB/everyday-conversations-llama3.1-2k",split="train")
if ec is not None:
    for idx,r in enumerate(ec):
        msgs=r.get("messages",[])
        if not isinstance(msgs,list): continue
        pairs=[]; pending_user=None
        for m in msgs:
            if not isinstance(m,dict): continue
            role=str(m.get("role",m.get("from",""))).lower()
            content=str(m.get("content",m.get("value",""))).strip()
            if not content: continue
            if role in ("user","human","prompter"):
                pending_user=content
            elif role in ("assistant","gpt") and pending_user:
                pairs.append((pending_user,content)); pending_user=None
        add_dialogue(chat_dialogues,pairs,"everyday_conversations",f"everyday_conversations:{idx}")

# 3) Dolly: single instruction-response examples are valid Q&A units.
dolly=try_load("databricks/databricks-dolly-15k",split="train")
if dolly is not None:
    for idx,r in enumerate(dolly):
        u=str(r.get("instruction","")).strip()
        ctx=str(r.get("context","")).strip()
        a=str(r.get("response","")).strip()
        if ctx: u += "\n\nContext:\n"+ctx
        if u and a:
            add_dialogue(qa_dialogues,[(u,a)],"dolly",f"dolly:{idx}")

# 4) OASST1: reconstruct all approved English assistant branches.
# Each assistant response is paired with its direct prompter parent. If an approved prior
# user/assistant turn exists, include it as context. All windows from a tree share one root ID.
oa=try_load("OpenAssistant/oasst1",split="train")
if oa is not None:
    rows=list(oa)
    byid={str(r.get("message_id")):r for r in rows if r.get("message_id") is not None}
    children={}
    for r in rows:
        parent=r.get("parent_id")
        if parent is not None:
            children.setdefault(str(parent),[]).append(r)

    def oa_approved(r, require_top_rank=False):
        if not r:
            return False
        if str(r.get("lang","")).lower() != "en" or r.get("deleted",False) is True:
            return False
        if r.get("review_result",True) is False:
            return False
        if require_top_rank:
            try:
                if int(r.get("rank",-1)) != 0:
                    return False
            except Exception:
                return False
        return True

    root_cache={}
    def oa_root_id(message_id):
        """Walk parent pointers to the top of this tree; cache root IDs."""
        cur=str(message_id)
        trail=[]
        seen=set()
        while cur in byid and cur not in root_cache and cur not in seen:
            seen.add(cur)
            trail.append(cur)
            parent=byid[cur].get("parent_id")
            if parent is None or str(parent) not in byid:
                root_cache[cur]=cur
                break
            cur=str(parent)
        root=root_cache.get(cur,cur)
        for node_id in trail:
            root_cache[node_id]=root
        return root

    for assistant in rows:
        if str(assistant.get("role","")).lower() != "assistant":
            continue
        if not oa_approved(assistant, require_top_rank=True):
            continue
        user_node=byid.get(str(assistant.get("parent_id")))
        if not user_node or str(user_node.get("role","")).lower() != "prompter":
            continue
        if not oa_approved(user_node):
            continue

        u_now=str(user_node.get("text","")).strip()
        a_now=str(assistant.get("text","")).strip()
        if not u_now or not a_now:
            continue

        pairs=[]
        # Include one prior turn if this prompt follows an approved English assistant
        # response, whose parent is an approved English prompter node.
        previous_assistant=byid.get(str(user_node.get("parent_id")))
        if previous_assistant and str(previous_assistant.get("role","")).lower() == "assistant" \
                and oa_approved(previous_assistant, require_top_rank=True):
            previous_user=byid.get(str(previous_assistant.get("parent_id")))
            if previous_user and str(previous_user.get("role","")).lower() == "prompter" \
                    and oa_approved(previous_user):
                u_prev=str(previous_user.get("text","")).strip()
                a_prev=str(previous_assistant.get("text","")).strip()
                if u_prev and a_prev:
                    pairs.append((u_prev,a_prev))

        pairs.append((u_now,a_now))
        root_id=oa_root_id(str(assistant.get("message_id")))
        add_dialogue(qa_dialogues,pairs,"oasst1",f"oasst1:{root_id}")

print("Full dialogue units:",len(chat_dialogues),"chat conversations;",len(qa_dialogues),"Q&A units")

def norm(s):
    s=unicodedata.normalize("NFKC",str(s)).lower()
    return re.sub(r"\s+"," ",s).strip()

def dialogue_hash(d):
    canonical="\n<turn>\n".join(norm(u)+"\n<assistant>\n"+norm(a) for u,a in d["turns"])
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

def encode_window(turns, source, conversation_id, start_turn):
    """Encode complete consecutive turns; only assistant responses + each EOS are loss targets."""
    ids=[]; labels=[]
    for u,a in turns:
        separator = "\\n" if ids else ""
        turn_prefix = separator + f"User: {u}\\nAssistant:"
        prefix=tokenizer.encode(turn_prefix,add_special_tokens=False)
        response=tokenizer.encode(" "+a,add_special_tokens=False)
        # A too-short or too-long response is not silently truncated.
        if len(response)<4: return None
        turn_ids=prefix+response+[EOS]
        turn_labels=[-100]*len(prefix)+response+[EOS]
        if len(ids)+len(turn_ids)>CFG["context"]: return None
        ids.extend(turn_ids); labels.extend(turn_labels)
    if not ids or len(ids)>CFG["context"] or sum(y!=-100 for y in labels)<4: return None
    return {"input_ids":ids,"labels":labels,"active":sum(y!=-100 for y in labels),
            "source":source,"conversation_id":str(conversation_id),
            "hash":hashlib.sha256((str(conversation_id)+":"+str(start_turn)+":"+
                 " ".join(map(str,ids))).encode()).hexdigest(),
            "turn_count":len(turns)}

def make_windows(dialogues, max_turns=2):
    """Greedy sliding windows of up to two turns, preserving complete assistant replies."""
    output=[]; rejected=defaultdict(int)
    for d in dialogues:
        turns=d["turns"]
        # Every start position gives a candidate history window; windows remain grouped by conversation_id.
        for start in range(len(turns)):
            selected=[]
            for j in range(start,min(len(turns),start+max_turns)):
                candidate=selected+[turns[j]]
                ex=encode_window(candidate,d["source"],d["conversation_id"],start)
                if ex is None:
                    if not selected: rejected["turn_too_long_or_short"]+=1
                    break
                selected=candidate
            if selected:
                ex=encode_window(selected,d["source"],d["conversation_id"],start)
                if ex: output.append(ex)
    print("Window filtering:",dict(rejected))
    return output

chat_windows=make_windows(chat_dialogues,max_turns=2)
# Keep up to two complete turns for OASST1 chains; single-turn Dolly items remain one turn.
qa_windows=make_windows(qa_dialogues,max_turns=2)
print("Candidate windows before dedup:",len(chat_windows),"chat;",len(qa_windows),"Q&A")

def dedup_windows(windows,name):
    seen=set(); out=[]; rejected=0
    for x in windows:
        # Content hash prevents exact duplicates; conversation_id is retained for group splitting.
        h=hashlib.sha256((norm(tokenizer.decode(x["input_ids"]))+"|"+str(x["turn_count"])).encode()).hexdigest()
        if h in seen: rejected+=1; continue
        seen.add(h); x["content_hash"]=h; out.append(x)
    print(name,"windows kept",len(out),"exact duplicates removed",rejected)
    return out
chat_windows=dedup_windows(chat_windows,"Chat")
qa_windows=dedup_windows(qa_windows,"Q&A")

def group_split(windows, validation_fraction=0.05, seed=20261017):
    groups={}
    for x in windows: groups.setdefault(x["conversation_id"],[]).append(x)
    ids=list(groups); random.Random(seed).shuffle(ids)
    target=max(1,int(len(windows)*validation_fraction))
    val_ids=set(); count=0
    for gid in ids:
        if count>=target: break
        val_ids.add(gid); count+=len(groups[gid])
    train=[x for x in windows if x["conversation_id"] not in val_ids]
    val=[x for x in windows if x["conversation_id"] in val_ids]
    return train,val,val_ids

chat_train, val_chat, chat_val_groups=group_split(chat_windows,seed=CFG["seed"]+1)
qa_train, val_qa, qa_val_groups=group_split(qa_windows,seed=CFG["seed"]+2)
assert not ({x["conversation_id"] for x in chat_train}&chat_val_groups)
assert not ({x["conversation_id"] for x in qa_train}&qa_val_groups)
assert not ({x["content_hash"] for x in chat_train}&{x["content_hash"] for x in val_chat})
assert not ({x["content_hash"] for x in qa_train}&{x["content_hash"] for x in val_qa})

# Require enough usable data to support meaningful ratios. Do not silently duplicate/up-sample.
if len(chat_train)<3000 or len(qa_train)<3000:
    raise RuntimeError(
        f"Safety gate failed: clean training windows chat={len(chat_train):,}, Q&A={len(qa_train):,}; "
        "each pool must have >= 3,000 examples. Check dataset loading and filtering."
    )

chat_capacity=sum(x["active"] for x in chat_train)
qa_capacity=sum(x["active"] for x in qa_train)
target=min(CFG["target_active_tokens_epoch"],int(chat_capacity/.70),int(qa_capacity/.70))
if target<CFG["min_active_tokens_epoch"]:
    raise RuntimeError(f"Insufficient common active-token budget: chat={chat_capacity:,}, Q&A={qa_capacity:,}, feasible={target:,}.")

def choose(pool,budget,seed):
    items=list(pool); random.Random(seed).shuffle(items); chosen=[]; total=0
    for x in items:
        if total>=budget: break
        chosen.append(x); total+=x["active"]
    return chosen,total
def build_mix(chat_ratio,seed):
    c,ct=choose(chat_train,int(target*chat_ratio),seed)
    q,qt=choose(qa_train,target-int(target*chat_ratio),seed+1)
    return c+q,c,q,ct,qt
mixA,chatA,qaA,cta,qta=build_mix(.70,CFG["seed"]+10)
mixB,chatB,qaB,ctb,qtb=build_mix(.30,CFG["seed"]+20)
random.Random(CFG["seed"]+30).shuffle(mixA); random.Random(CFG["seed"]+31).shuffle(mixB)

def summary(name,c,q,ct,qt):
    total=ct+qt
    print(f"{name}: chat={len(c)} examples/{ct:,} active tokens ({ct/max(total,1):.1%}); "
          f"Q&A={len(q)} examples/{qt:,} ({qt/max(total,1):.1%}); total={total:,}")
summary("Mix A",chatA,qaA,cta,qta); summary("Mix B",chatB,qaB,ctb,qtb)
print("Validation:",len(val_chat),"chat windows +",len(val_qa),"Q&A windows; common active-token budget:",f"{target:,}")
for pool in (mixA,mixB,val_chat,val_qa):
    for x in pool:
        assert len(x["input_ids"])<=CFG["context"] and len(x["input_ids"])==len(x["labels"])
        assert x["labels"][-1]==EOS and sum(y!=-100 for y in x["labels"])>=4
        assert all(y==-100 for y in x["labels"][:next(i for i,y in enumerate(x["labels"]) if y!=-100)])
assert not ({x["conversation_id"] for x in mixA}&chat_val_groups)
assert not ({x["conversation_id"] for x in mixB}&chat_val_groups)
assert not ({x["conversation_id"] for x in mixA}&qa_val_groups)
assert not ({x["conversation_id"] for x in mixB}&qa_val_groups)

def inspect(x):
    print("\nSOURCE:",x["source"],"| turns:",x["turn_count"],"| conversation:",x["conversation_id"])
    print("MASKED PREFIX TOKENS:",tokenizer.decode([i for i,y in zip(x["input_ids"],x["labels"]) if y==-100]))
    print("LOSS TARGETS:",tokenizer.decode([i for i,y in zip(x["input_ids"],x["labels"]) if y!=-100]))
    print("length:",len(x["input_ids"]),"active:",x["active"],"EOS:",x["labels"][-1])
inspect(mixA[0]); inspect(mixB[0])
notify("TinyGPT data ready",f"chat windows={len(chat_train)} QA windows={len(qa_train)}; budget={target:,}; MixA={cta+qta:,}; MixB={ctb+qtb:,}")

Parquet load failed for daily_dialog: HfUriError: Invalid HF URI 'hf://datasets/daily_dialog@refs/convert/parquet/default/train/*.parquet'. Repository id must be 'namespace/name', got 'daily_dialog'.


default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 3.61MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Loaded parquet: li2017dailydialog/daily_dialog | columns=['dialog', 'act', 'emotion'] | rows=11118


...

data/validation-00000-of-00001-134b8fd0c(…): reconstructing file:   0%|          |  0.00B / 2.08MB            

data/validation-00000-of-00001-134b8fd0c(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/84437 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4401 [00:00<?, ? examples/s]

Loaded OpenAssistant/oasst1 columns= ['message_id', 'parent_id', 'user_id', 'created_date', 'text', 'role', 'lang', 'review_count', 'review_result', 'deleted', 'rank', 'synthetic', 'model_name', 'detoxify', 'message_tree_id', 'tree_state', 'emojis', 'labels'] rows= 84437
Full dialogue units: 13378 chat conversations; 22841 Q&A units
Window filtering: {'turn_too_long_or_short': 1898}
Window filtering: {'turn_too_long_or_short': 7563}
Candidate windows before dedup: 48364 chat; 16779 Q&A
Chat windows kept 44983 exact duplicates removed 3381
Q&A windows kept 16271 exact duplicates removed 508
Mix A: chat=25239 examples/840,038 active tokens (70.0%); Q&A=4892 examples/360,053 (30.0%); total=1,200,091
Mix B: chat=10811 examples/360,027 active tokens (30.0%); Q&A=11398 examples/840,043 (70.0%); total=1,200,070
Validation: 2251 chat windows + 813 Q&A windows; common active-token budget: 1,200,000

SOURCE: daily_dialog | turns: 2 | conversation: daily_dialog:1200
MASKED PREFIX TOKENS: User: Ha

## Cell 3 — Training engine and Mix A

In [3]:
class TensorSFT(Dataset):
    def __init__(self,records): self.records=records
    def __len__(self): return len(self.records)
    def __getitem__(self,i):
        x=self.records[i]; n=len(x["input_ids"]); pad=256-n
        return {"input_ids":torch.tensor(x["input_ids"]+[EOS]*pad),
                "labels":torch.tensor(x["labels"]+[-100]*pad),
                "attention_mask":torch.tensor([1]*n+[0]*pad)}
def collate(rows): return {k:torch.stack([r[k] for r in rows]) for k in rows[0]}

class LossSumModel(nn.Module):
    def __init__(self,model): super().__init__(); self.model=model
    def forward(self,input_ids,attention_mask,labels):
        logits=self.model(input_ids=input_ids,attention_mask=attention_mask,use_cache=False).logits
        sl=logits[:,:-1,:].contiguous(); sy=labels[:,1:].contiguous()
        loss_sum=F.cross_entropy(sl.reshape(-1,sl.size(-1)),sy.reshape(-1),ignore_index=-100,reduction="sum")
        count=(sy!=-100).sum().to(loss_sum.dtype)
        return torch.stack((loss_sum,count))

def atomic_save(obj,path):
    tmp=str(path)+".tmp"; torch.save(obj,tmp); os.replace(tmp,path)
def gpu_mem():
    return " | ".join([f"GPU{i} {torch.cuda.memory_allocated(i)/1024**3:.2f}/{torch.cuda.memory_reserved(i)/1024**3:.2f} GiB"
                       for i in range(torch.cuda.device_count())])
def sft_loss(model,records,batch_size=16):
    model.eval(); total=0.; count=0
    dl=DataLoader(TensorSFT(records),batch_size=batch_size,shuffle=False,collate_fn=collate)
    with torch.inference_mode():
        for b in dl:
            ids=b["input_ids"].to(device); labels=b["labels"].to(device); mask=b["attention_mask"].to(device)
            with torch.autocast("cuda",dtype=torch.float16):
                logits=model(input_ids=ids,attention_mask=mask,use_cache=False).logits[:,:-1,:].contiguous()
                sy=labels[:,1:].contiguous()
                ls=F.cross_entropy(logits.reshape(-1,logits.size(-1)),sy.reshape(-1),ignore_index=-100,reduction="sum")
            total+=float(ls); count+=int((sy!=-100).sum())
    return total/max(count,1)
def metrics(model):
    return {"val_loss_chat":sft_loss(model,val_chat),"val_loss_qa":sft_loss(model,val_qa),"val_loss_fineweb":fineweb_loss(model)}

def run_mix(name,records,base_state):
    seed=CFG["seed"]+(101 if name=="mixA" else 202); seed_all(seed)
    out=Path(CFG["out"])/name; out.mkdir(parents=True,exist_ok=True)
    best_path=out/f"sft_{name}_best.pt"; latest_path=out/f"sft_{name}_latest.pt"; final_path=out/f"sft_{name}_final.pt"
    model=GPT2LMHeadModel(MC); model.load_state_dict(base_state,strict=True); model.to(device)
    wrapped=LossSumModel(model)
    if torch.cuda.device_count()>1: wrapped=nn.DataParallel(wrapped)
    batch_size=16*max(torch.cuda.device_count(),1)
    dl=DataLoader(TensorSFT(records),batch_size=batch_size,shuffle=True,collate_fn=collate,num_workers=0,
                  generator=torch.Generator().manual_seed(seed))
    steps_epoch=math.ceil(len(dl)/CFG["grad_accum"]); total_steps=steps_epoch*CFG["epochs"]
    warm=max(1,int(total_steps*CFG["warmup_fraction"]))
    opt=torch.optim.AdamW(model.parameters(),lr=CFG["lr"],weight_decay=CFG["weight_decay"],betas=(.9,.95))
    sched=get_cosine_schedule_with_warmup(opt,warm,total_steps)
    scaler=torch.amp.GradScaler("cuda")
    best=float("inf"); step=0; active_seen=0; examples_seen=0; start=time.time(); roll_t=start; roll_tok=roll_ex=0
    opt.zero_grad(set_to_none=True); notify(name+" started",f"examples={len(records)} steps={total_steps}")
    for ep in range(CFG["epochs"]):
        model.train(); pending=[]
        for bi,b in enumerate(dl):
            pending.append(b)
            if len(pending)<CFG["grad_accum"] and bi<len(dl)-1: continue
            for mb in pending:
                ids=mb["input_ids"].to(device); labels=mb["labels"].to(device); mask=mb["attention_mask"].to(device)
                with torch.autocast("cuda",dtype=torch.float16):
                    got=wrapped(input_ids=ids,attention_mask=mask,labels=labels)
                    if got.ndim==2: ls=got[:,0].sum(); cnt=got[:,1].sum()
                    else: ls,cnt=got[0],got[1]
                    loss=ls/cnt.clamp_min(1.)/len(pending)
                scaler.scale(loss).backward()
                n=int((labels[:,1:]!=-100).sum()); active_seen+=n; roll_tok+=n; roll_ex+=ids.shape[0]; examples_seen+=ids.shape[0]
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),CFG["grad_clip"])
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            for g in opt.param_groups: g["lr"]=max(g["lr"],CFG["min_lr"])
            step+=1; pending=[]
            if step%CFG["log_every"]==0 or step==1:
                now=time.time(); elapsed=now-start; rate=roll_tok/max(now-roll_t,1e-6); eta=max(total_steps-step,0)*elapsed/max(step,1)
                print(f"[{name}] epoch {ep+1}/{CFG['epochs']} step {step}/{total_steps} examples={examples_seen} active_tokens={active_seen:,} "
                      f"active_tok/s={rate:.0f} examples/s={roll_ex/max(now-roll_t,1e-6):.1f} lr={opt.param_groups[0]['lr']:.2e} "
                      f"elapsed={elapsed/60:.1f}m ETA={int(eta//3600)}h{int(eta%3600//60):02d}m | {gpu_mem()}")
                roll_t=now; roll_tok=roll_ex=0
                if step%max(CFG["log_every"]*4,1)==0: notify(name+" progress",f"step {step}/{total_steps}; active tok/s={rate:.0f}; ETA {eta/60:.1f} min")
            if step%CFG["validate_every"]==0 or step==total_steps:
                met=metrics(model); score=met["val_loss_chat"]+met["val_loss_qa"]
                print(f"[{name}] validation",met)
                if score<best:
                    best=score; atomic_save(model.state_dict(),best_path)
                atomic_save({"model_state_dict":model.state_dict(),"optimizer_state_dict":opt.state_dict(),
                             "scheduler_state_dict":sched.state_dict(),"scaler_state_dict":scaler.state_dict(),
                             "epoch":ep,"step":step,"metrics":met,"config":CFG},latest_path)
                notify(name+" validation",f"step={step}; {met}")
                model.train()
    atomic_save(model.state_dict(),final_path)
    met=metrics(model); print(name,"final metrics",met)
    notify(name+" complete",str(met))
    del wrapped,model,opt,sched,scaler,dl; gc.collect(); torch.cuda.empty_cache()
    return {"best":str(best_path),"final":str(final_path),"latest":str(latest_path),"metrics":met}

mixA_result=run_mix("mixA",mixA,BASE_STATE)
print("Mix A outputs:",mixA_result)


[mixA] epoch 1/2 step 1/942 examples=64 active_tokens=2,714 active_tok/s=1222 examples/s=28.8 lr=5.00e-06 elapsed=0.0m ETA=0h34m | GPU0 0.80/4.66 GiB | GPU1 0.02/5.14 GiB
[mixA] epoch 1/2 step 25/942 examples=1600 active_tokens=62,928 active_tok/s=5486 examples/s=140.0 lr=2.66e-05 elapsed=0.2m ETA=0h08m | GPU0 0.80/5.43 GiB | GPU1 0.02/5.14 GiB
[mixA] epoch 1/2 step 50/942 examples=3200 active_tokens=125,589 active_tok/s=5448 examples/s=139.1 lr=5.00e-05 elapsed=0.4m ETA=0h07m | GPU0 0.80/5.43 GiB | GPU1 0.02/5.14 GiB
[mixA] epoch 1/2 step 75/942 examples=4800 active_tokens=189,074 active_tok/s=5503 examples/s=138.7 lr=4.99e-05 elapsed=0.6m ETA=0h06m | GPU0 0.80/5.43 GiB | GPU1 0.02/5.14 GiB
[mixA] epoch 1/2 step 100/942 examples=6400 active_tokens=253,819 active_tok/s=5601 examples/s=138.4 lr=4.96e-05 elapsed=0.8m ETA=0h06m | GPU0 0.80/5.43 GiB | GPU1 0.02/5.14 GiB
[mixA] validation {'val_loss_chat': 3.17841772104334, 'val_loss_qa': 3.2868184394159727, 'val_loss_fineweb': 3.7193951567

## Cell 4 — Mix B from fresh Stage 3 weights

In [4]:
gc.collect(); torch.cuda.empty_cache()
# New model and new optimizer are constructed inside run_mix; only BASE_STATE is reused.
mixB_result=run_mix("mixB",mixB,BASE_STATE)
print("Mix B outputs:",mixB_result)
notify("TinyGPT dual SFT complete",f"Mix A {mixA_result['metrics']}; Mix B {mixB_result['metrics']}")


[mixB] epoch 1/2 step 1/696 examples=64 active_tokens=2,961 active_tok/s=3620 examples/s=78.3 lr=5.00e-06 elapsed=0.0m ETA=0h09m | GPU0 0.80/5.37 GiB | GPU1 0.02/4.40 GiB
[mixB] epoch 1/2 step 25/696 examples=1600 active_tokens=90,108 active_tok/s=7675 examples/s=135.3 lr=3.68e-05 elapsed=0.2m ETA=0h05m | GPU0 0.80/5.38 GiB | GPU1 0.02/4.40 GiB
[mixB] epoch 1/2 step 50/696 examples=3200 active_tokens=176,830 active_tok/s=7303 examples/s=134.7 lr=4.99e-05 elapsed=0.4m ETA=0h05m | GPU0 0.80/5.38 GiB | GPU1 0.02/4.40 GiB
[mixB] epoch 1/2 step 75/696 examples=4800 active_tokens=262,361 active_tok/s=7212 examples/s=134.9 lr=4.95e-05 elapsed=0.6m ETA=0h04m | GPU0 0.80/5.38 GiB | GPU1 0.02/4.40 GiB
[mixB] epoch 1/2 step 100/696 examples=6400 active_tokens=349,676 active_tok/s=7384 examples/s=135.3 lr=4.88e-05 elapsed=0.8m ETA=0h04m | GPU0 0.80/5.38 GiB | GPU1 0.02/4.40 GiB
[mixB] validation {'val_loss_chat': 3.245894456212023, 'val_loss_qa': 3.224521449019175, 'val_loss_fineweb': 3.7192181555

## Cell 5 — Base vs Mix A vs Mix B evaluation and reports

In [5]:
PROMPTS=[
("chat","Hi! How has your day been?"),("chat","I had a difficult day today."),
("chat","What do you like to do in your free time?"),("chat","I'm excited about learning something new."),
("chat","Can you help me think through a problem?"),("chat","That sounds interesting. Tell me more."),
("chat","I don't know what to do next."),("chat","Thanks for explaining that."),
("chat","What would you recommend for a relaxing evening?"),("chat","Let's talk about technology."),
("qa","What is supervised fine-tuning?"),("qa","Explain what a GPU does in simple terms."),
("qa","Write three benefits of exercise."),("qa","What is the capital of France?"),
("qa","At standard atmospheric pressure, what is the boiling point of water in Celsius?"),
("qa","What is photosynthesis?"),("qa","Write a Python function that returns the square of a number."),
("qa","Give two differences between an HDD and an SSD."),("qa","What is the difference between a CPU and a GPU?"),
("qa","A person has 12 stones, gets 8 more, then gives away 5. How many remain?")]
FACTS=[p for cat,p in PROMPTS if cat=="qa"]
def gen(model,prompt,seed=11,sample=True):
    enc=tokenizer(f"User: {prompt}\nAssistant:",return_tensors="pt",truncation=True,max_length=220).to(device)
    if sample: torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    args={"max_new_tokens":CFG["max_new_tokens"],"do_sample":sample,"repetition_penalty":1.15,
          "eos_token_id":EOS,"pad_token_id":EOS,"use_cache":True}
    if sample: args.update({"temperature":.7,"top_p":.9})
    model.eval()
    with torch.inference_mode(): out=model.generate(**enc,**args)
    return tokenizer.decode(out[0,enc["input_ids"].shape[1]:],skip_special_tokens=True).strip()
def load_best(path):
    st=torch.load(path,map_location="cpu",weights_only=True)
    m=GPT2LMHeadModel(MC); m.load_state_dict(st,strict=True); return m.to(device)
models={"BASE":new_model(BASE_STATE).to(device),
        "MIX_A":load_best(mixA_result["best"]),
        "MIX_B":load_best(mixB_result["best"])}
report={"baseline_fineweb_loss":base_loss,"mixA_training":mixA_result,"mixB_training":mixB_result,
        "metrics":{},"greedy_factual":{},"sampled_benchmark":{}}
for name,m in models.items():
    report["metrics"][name]=metrics(m)
    report["greedy_factual"][name]=[{"prompt":p,"response":gen(m,p,sample=False)} for p in FACTS]
    rows=[]
    for cat,p in PROMPTS:
        for seed in CFG["seeds"]:
            rows.append({"category":cat,"prompt":p,"seed":seed,"response":gen(m,p,seed=seed,sample=True)})
    report["sampled_benchmark"][name]=rows
out=Path(CFG["out"]); out.mkdir(parents=True,exist_ok=True)
jp=out/"sft_comparison_report.json"; tp=out/"sft_comparison_report.txt"
jp.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
with tp.open("w",encoding="utf-8") as f:
    f.write("TinyGPT v2 SFT comparison: Base vs Mix A vs Mix B\n\n")
    f.write("METRICS\n")
    for n,v in report["metrics"].items(): f.write(f"{n}: {json.dumps(v)}\n")
    f.write("\nGREEDY FACTUAL PROBE\n")
    for n,rows in report["greedy_factual"].items():
        f.write(f"\n--- {n} ---\n")
        for r in rows: f.write(f"Q: {r['prompt']}\nA: {r['response']}\n\n")
    f.write("\nSAMPLED BENCHMARK\n")
    for n,rows in report["sampled_benchmark"].items():
        f.write(f"\n--- {n} ---\n")
        for r in rows: f.write(f"[{r['category']}] seed={r['seed']} Q: {r['prompt']}\nA: {r['response']}\n\n")
print("Final metrics:\n",json.dumps(report["metrics"],indent=2))
print("Reports:",tp,"and",jp)
notify("TinyGPT SFT report ready",f"Reports saved in {out}; metrics={report['metrics']}")
del models; gc.collect(); torch.cuda.empty_cache()


Final metrics:
 {
  "BASE": {
    "val_loss_chat": 4.122481818129509,
    "val_loss_qa": 3.491162043031319,
    "val_loss_fineweb": 3.644027994501432
  },
  "MIX_A": {
    "val_loss_chat": 3.0027391575534814,
    "val_loss_qa": 3.3007397728372587,
    "val_loss_fineweb": 3.8822881562926996
  },
  "MIX_B": {
    "val_loss_chat": 3.1022686530374632,
    "val_loss_qa": 3.1911950133682927,
    "val_loss_fineweb": 3.8098594378546538
  }
}
Reports: /kaggle/working/tinygpt_sft/sft_comparison_report.txt and /kaggle/working/tinygpt_sft/sft_comparison_report.json
